# Day 2 — RAG chain + guardrail

Loads the FAISS index built in `01_ingest.ipynb`, wires up retrieval → local LLM →
citation-formatted answer, then adds two guardrails that make the agent decline
rather than answer when it shouldn't:

1. **Confidence guardrail** — decline when retrieval similarity is too weak
   (the question probably isn't covered by the corpus at all).
2. **Scope guardrail** — decline specific topics the WHO guideline explicitly
   excludes (hypertensive emergencies/urgencies, drug dosing specifics,
   secondary/resistant hypertension) even if retrieval finds something that
   *looks* related — confidence alone can't catch these.

## Piece 1 — Load the saved index

Reuse the same `OllamaEmbeddings` model as Day 1 (the query has to be embedded
with the same model the chunks were embedded with, or the vectors aren't
comparable) and reload the FAISS index from disk instead of rebuilding it.

In [10]:
from pathlib import Path

from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OllamaEmbeddings(model="nomic-embed-text")

index_path = Path("..") / "data" / "faiss_index"
vectorstore = FAISS.load_local(
    str(index_path), embeddings, allow_dangerous_deserialization=True
)

print(f"Loaded FAISS index with {vectorstore.index.ntotal} vectors")

Loaded FAISS index with 108 vectors


## Piece 2 — Retrieval → LLM → answer

`langchain_community.llms.Ollama` (not `ChatOllama`) because that's what's
actually installed — `langchain-ollama` isn't in `requirements.txt`, and this
plain-completion wrapper does everything a single-turn Q&A prompt needs, so
there's no reason to add a new dependency for it.

The prompt explicitly instructs the model to answer *only* from the retrieved
context and to say so if the context doesn't cover the question — this is a
second, prompt-level line of defense underneath the score-based guardrail in
Piece 4 (which runs *before* the LLM is even called).

In [11]:
from langchain_community.llms import Ollama

llm = Ollama(model="llama3.2:3b")

ANSWER_PROMPT = """You are a clinical guideline assistant. Answer the question \
using ONLY the context below, which is excerpted from public CDC/WHO/USPSTF \
hypertension guidelines. Do not use outside knowledge. If the context does not \
contain enough information to answer, say so plainly instead of guessing.

This is informational only — not diagnostic or treatment advice for any
individual patient.

Context:
{context}

Question: {question}

Answer:"""


def retrieve(question: str, k: int = 4):
    """Top-k (Document, score) pairs; lower score = more similar (FAISS L2 distance)."""
    return vectorstore.similarity_search_with_score(question, k=k)


def generate_answer(question: str, retrieved) -> str:
    context = "\n\n---\n\n".join(doc.page_content for doc, _ in retrieved)
    prompt = ANSWER_PROMPT.format(context=context, question=question)
    return llm.invoke(prompt)


# Quick check before adding citations/guardrails in the next pieces.
test_question = "What blood pressure threshold should trigger starting medication?"
test_retrieved = retrieve(test_question)
print(generate_answer(test_question, test_retrieved))

According to the guidelines, the blood pressure thresholds that trigger starting medication are as follows:

- Systolic blood pressure (SBP) of ≥140 mmHg or diastolic blood pressure (DBP) of ≥90 mmHg.
- For individuals with existing cardiovascular disease, the threshold is a systolic blood pressure of 130-139 mmHg.
- For individuals without cardiovascular disease but with high cardiovascular risk, diabetes mellitus, or chronic kidney disease, the threshold is also a systolic blood pressure of 130-139 mmHg.


## Piece 3 — Inline citations

Every chunk already carries `title`, `publisher`, and `page_label` metadata
(stamped in Day 1). Append a dedupe'd "Sources" list built from the actual
retrieved chunks — not something the LLM writes itself — so citations can't
be hallucinated or mismatched to the wrong page.

In [12]:
def format_citations(retrieved) -> str:
    seen = []
    for doc, _ in retrieved:
        m = doc.metadata
        entry = f"{m['title']} ({m['publisher']}), p. {m['page_label']}"
        if entry not in seen:
            seen.append(entry)
    return "\n".join(f"[{i+1}] {c}" for i, c in enumerate(seen))


def answer_with_citations(question: str, k: int = 4) -> str:
    retrieved = retrieve(question, k=k)
    answer = generate_answer(question, retrieved)
    citations = format_citations(retrieved)
    return f"{answer}\n\nSources:\n{citations}"


print(answer_with_citations("What does USPSTF recommend for screening adults for high blood pressure?"))

The USPSTF recommends screening for hypertension in adults 18 years or older without known hypertension.

Sources:
[1] Screening for Hypertension in Adults (U.S. Preventive Services Task Force), p. 4
[2] Screening for Hypertension in Adults (U.S. Preventive Services Task Force), p. 2
[3] Screening for Hypertension in Adults (U.S. Preventive Services Task Force), p. 1
[4] Screening for Hypertension in Adults (U.S. Preventive Services Task Force), p. 3


## Piece 4 — Confidence guardrail

FAISS's `similarity_search_with_score` returns an L2 distance — **lower is
more similar** (this is the opposite of a 0–1 similarity percentage, easy to
get backwards). To pick a threshold empirically, run a few genuinely
answerable questions (covered by the corpus) and a few genuinely unanswerable
ones (unrelated to hypertension entirely) through retrieval and look at where
the scores split.

Ran this once outside the notebook first: answerable top-1 scores landed in
the 196–276 range, unanswerable top-1 scores were all 368+. That's a wide,
clean gap — pick a threshold roughly in the middle of it. **A score above the
threshold means "not similar enough" → decline.**

In [13]:
answerable_test_qs = [
    "What blood pressure threshold should trigger starting medication?",
    "What does USPSTF recommend for screening adults for high blood pressure?",
    "What are some strategies to help patients manage hypertension at a practice level?",
]
unanswerable_test_qs = [
    "What is the recommended chemotherapy regimen for stage 4 lung cancer?",
    "How do I fix a null pointer exception in Java?",
    "What is the capital of France?",
]

print("--- answerable (top-1 score) ---")
for q in answerable_test_qs:
    score = retrieve(q, k=1)[0][1]
    print(f"{score:7.1f}  {q}")

print("\n--- unanswerable (top-1 score) ---")
for q in unanswerable_test_qs:
    score = retrieve(q, k=1)[0][1]
    print(f"{score:7.1f}  {q}")

--- answerable (top-1 score) ---
  231.4  What blood pressure threshold should trigger starting medication?
  195.7  What does USPSTF recommend for screening adults for high blood pressure?
  268.3  What are some strategies to help patients manage hypertension at a practice level?

--- unanswerable (top-1 score) ---
  368.5  What is the recommended chemotherapy regimen for stage 4 lung cancer?
  515.9  How do I fix a null pointer exception in Java?
  548.6  What is the capital of France?


In [14]:
# Midpoint of the gap observed above (~276 to ~368). Not tuned further than
# that — the gap is wide enough that the exact midpoint doesn't matter much.
CONFIDENCE_THRESHOLD = 320.0

DECLINE_LOW_CONFIDENCE = (
    "I don't have enough information in the corpus (WHO pharmacological "
    "hypertension guideline, USPSTF screening recommendation, and CDC Million "
    "Hearts change package) to answer that confidently. This tool only covers "
    "adult hypertension screening and management, and is informational only "
    "— not diagnostic or treatment advice."
)


def is_low_confidence(retrieved, threshold: float = CONFIDENCE_THRESHOLD) -> bool:
    top_score = retrieved[0][1]
    return top_score > threshold

## Piece 5 — Scope guardrail

Confidence alone won't catch everything: a question about, say, dosing a
specific antihypertensive drug can retrieve a *highly* similar chunk (the
corpus mentions drug classes) while still being outside what this tool should
answer — per the 2026-08-31 scope decision, the WHO guideline explicitly
excludes hypertensive emergencies/urgencies, drug dosing specifics, and
secondary/resistant hypertension. A keyword check catches these regardless of
retrieval confidence, and runs *before* retrieval so an excluded question
never even reaches the LLM.

In [15]:
import re

# Keyword patterns for the three excluded topics (2026-08-31 scope decision).
# Not exhaustive medical terminology — covers the common ways someone would
# actually phrase these questions in a demo.
#
# The "drug dosing specifics" pattern requires a digit before "mg" (an
# actual dose amount, e.g. "12.5 mg") rather than matching bare "mg" —
# testing turned up a false positive on in-scope questions like sodium
# intake ("...recommended in mg per day"), which is a dietary unit covered
# by the CDC corpus, not a drug dose.
SCOPE_EXCLUSIONS = {
    "hypertensive emergency/urgency": re.compile(
        r"hypertensive (emergenc|urgenc)|\bmalignant hypertension\b", re.I
    ),
    "drug dosing specifics": re.compile(
        r"\bdos(e|age|ing)\b|\d+\s*mg\b|\bmilligram", re.I
    ),
    "secondary/resistant hypertension": re.compile(
        r"secondary hypertension|resistant hypertension", re.I
    ),
}


def out_of_scope_topic(question: str) -> str | None:
    """Returns the matched exclusion label, or None if in scope."""
    for label, pattern in SCOPE_EXCLUSIONS.items():
        if pattern.search(question):
            return label
    return None


def decline_scope_message(label: str) -> str:
    return (
        f"This question touches on {label}, which the underlying WHO "
        "guideline explicitly excludes from its scope, so this tool declines "
        "to answer it. This is informational only — not diagnostic or "
        "treatment advice."
    )


# Sanity check: the three excluded topics, phrased a few ways, plus edge
# cases that should NOT trigger a decline — including the sodium-intake
# question that exposed the bare-"mg" false positive above.
for q in [
    "What should I do for a hypertensive emergency?",
    "What is the dosage of lisinopril for hypertension?",
    "How is resistant hypertension managed?",
    "What blood pressure threshold should trigger starting medication?",
    "How much sodium, in mg, is recommended per day for someone managing hypertension?",
    "What magnesium-related dietary changes are recommended for blood pressure?",
    "Is it urgent to treat stage 1 hypertension immediately?",
]:
    print(f"{out_of_scope_topic(q)!r:45} {q}")

'hypertensive emergency/urgency'              What should I do for a hypertensive emergency?
'drug dosing specifics'                       What is the dosage of lisinopril for hypertension?
'secondary/resistant hypertension'            How is resistant hypertension managed?
None                                          What blood pressure threshold should trigger starting medication?
None                                          How much sodium, in mg, is recommended per day for someone managing hypertension?
None                                          What magnesium-related dietary changes are recommended for blood pressure?
None                                          Is it urgent to treat stage 1 hypertension immediately?


## Piece 6 — Put it together: `ask()`

Order matters: scope check first (cheapest, and catches things confidence
wouldn't), then retrieval + confidence check, then — only if both pass — the
LLM call and citation formatting. This means an excluded or off-corpus
question never reaches the (slowest, ~10s) LLM call at all.

In [16]:
def ask(question: str, k: int = 4) -> str:
    scope_hit = out_of_scope_topic(question)
    if scope_hit:
        return decline_scope_message(scope_hit)

    retrieved = retrieve(question, k=k)
    if is_low_confidence(retrieved):
        return DECLINE_LOW_CONFIDENCE

    answer = generate_answer(question, retrieved)
    citations = format_citations(retrieved)
    return f"{answer}\n\nSources:\n{citations}"

## End-to-end check

One question per path — normal answer, scope decline, confidence decline —
so all three behaviors are demonstrated together (this is also the set worth
reusing in the Day 3 README as example Q&A).

In [17]:
demo_questions = [
    "What blood pressure threshold should trigger starting pharmacological treatment?",
    "What is the dosage of amlodipine for hypertension?",
    "What is the recommended chemotherapy regimen for stage 4 lung cancer?",
]

for q in demo_questions:
    print(f"Q: {q}\n")
    print(ask(q))
    print("\n" + "=" * 80 + "\n")

Q: What blood pressure threshold should trigger starting pharmacological treatment?

According to the guidelines, the blood pressure threshold that should trigger starting pharmacological treatment is:

* ≥140 mmHg (systolic blood pressure) or ≥90 mmHg (diastolic blood pressure) for adults without cardiovascular disease, diabetes mellitus, or chronic kidney disease.
* ≥130 mmHg (systolic blood pressure) for adults with cardiovascular disease, diabetes mellitus, or chronic kidney disease.

Note that there are different thresholds for starting pharmacological treatment in patients with high cardiovascular risk, diabetes mellitus, or chronic kidney disease.

Sources:
[1] Guideline for the pharmacological treatment of hypertension in adults (World Health Organization), p. 7
[2] Guideline for the pharmacological treatment of hypertension in adults (World Health Organization), p. 26
[3] Guideline for the pharmacological treatment of hypertension in adults (World Health Organization), p. 10
[